# EDA RC Energía

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from statsmodels.tsa.stattools import acf

In [ ]:
BASE_DIR = Path.cwd()  # notebooks/
PROJECT_ROOT = BASE_DIR.parent  # notebooks/ -> raíz del repo
sys.dont_write_bytecode = True  # no crear __pycache__ en data_utils/ (alcance: solo leer)
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from data_utils.cat_num_dep import hub_pruebas_num_cat
from data_utils.fancy_barplot import fancy_bars
from data_utils.fancy_hist import fancy_hist
from data_utils.triple_plot import normality_report
from data_utils.vif_resume import vif_table

In [ ]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="white")
INK, GRID = "#1F2E3D", "#E2E8F0"

# ---------------- Parámetros ----------------
NOMBRE_EXCEL = "Base de datos.xlsx"
HOJA, POSICION_HOJA = "RC ENERGIA", 0
RANDOM_STATE = 42
TEST_SIZE = 0.20

COL_TIEMPO = "t_stamp"
PROCESO = ["FILTRO 1", "FILTRO 2", "HORNO", "REFINO 1", "REFINO 2", "TRITURADO", "PTARI"]
NUEVOS = ["EMCALI", "EDIF ADMON"]
MEDIDORES = NUEVOS + PROCESO
ESQUEMA = {COL_TIEMPO: "datetime", **dict.fromkeys(MEDIDORES, "float64")}  # tipo lógico esperado

TURNOS = {"T1": "22:00-06:00", "T2": "06:00-14:00", "T3": "14:00-22:00"}
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

In [ ]:
ALPHA = 0.05  # nivel de significancia
PASO_MIN_H, PASO_MAX_H = 0.95, 1.5  # paso normal entre lecturas (h)
Z_MAD_UMBRAL = 3.5  # umbral de outlier robusto
MIN_HORAS_VALIDAS_DIA = 12  # horas mínimas para promediar un día
PCT_ESTRUCTURAL, PCT_REVISAR = 50, 5  # umbrales de la tabla de salud (%)

## 1. Carga de la hoja RC ENERGIA

In [ ]:
RUTA = Path("../data/Base de datos.xlsx")
raw = pd.read_excel(RUTA, sheet_name=HOJA)
print(f"Libro: {RUTA.name}")
print(f"\nhojas: {pd.ExcelFile(RUTA).sheet_names}")
print(f"\nHoja cargada: '{HOJA}'")
print(f"\nforma: {raw.shape}")
dtypes_crudos = raw.dtypes.astype(str)

## 2. Split train/test

Antes de separar los datos, aplicamos transformaciones que requieren el dataset completo y ordenado cronológicamente para no romper la serie de tiempo. El proceso sigue estos pasos:

- Formato y orden cronológico: Se validan los tipos de datos (fechas y números) registrando cualquier celda defectuosa, y se ordena el dataset desde la lectura más antigua a la más reciente.

- Cálculo de consumo real (Deltas): Dado que los medidores entregan valores acumulados, se calcula la diferencia de consumo y de tiempo entre cada fila y su lectura anterior.

- Detección de anomalías en sensores: Se crean marcas para identificar periodos donde los medidores se "congelaron" (sin registrar cambios) y el momento exacto en el que vuelven a reaccionar.

- División segura y aislamiento: Se realiza el split y el conjunto de prueba (test) se elimina deliberadamente de la memoria. Esto asegura que todo el análisis estadístico se haga solo con train, previniendo por completo la fuga de datos (data leakage).

> **Nota para el modelado:** con `SHUFFLE=True` el train y el test se mezclan en el tiempo. Para el EDA eso es aceptable, pero para un modelo de **pronóstico** la evaluación debe ser cronológica (`SHUFFLE=False`; en ese caso sklearn ignora el `random_state`). Basta con cambiar el parámetro.

In [ ]:
X = raw.copy()

fallo_cast = {}  # donde se generan nulos por errores de conversión de tipo
X[COL_TIEMPO] = pd.to_datetime(
    X[COL_TIEMPO], errors="coerce"
)  # convierte a datetime, los errores se convierten en NaN
fallo_cast[COL_TIEMPO] = (
    X[COL_TIEMPO].isna() & raw[COL_TIEMPO].notna()
)  # Detecta errores de conversión a datetime
# si el valor original no era nulo y el convertido sí lo es
# entonces hubo un error de conversión

for m in MEDIDORES:
    num = pd.to_numeric(X[m], errors="coerce").astype("float64")
    fallo_cast[m] = num.isna() & raw[m].notna()
    X[m] = num
for c, s in fallo_cast.items():
    X[f"cast_falla__{c}"] = s

X = X.sort_values(COL_TIEMPO, kind="mergesort").reset_index(
    drop=True
)  # Ordena por tiempo de la fecha más antigua a la más reciente
X["paso_h"] = (
    X[COL_TIEMPO].diff().dt.total_seconds() / 3600
)  # calcula la diferencia de tiempo entre cada fila en horas

for m in MEDIDORES:
    X[f"prev__{m}"] = X[m].shift(1)
    X[f"delta__{m}"] = (
        X[m] - X[f"prev__{m}"]
    )  # consumo real en el intervalo de tiempo entre cada fila
X["congelado_general"] = (
    X[[f"delta__{m}" for m in PROCESO]].eq(0).all(axis=1)
)  # verifica si todos los procesos tienen delta 0 (fallo de medición o apagado de planta)
X["tras_congelamiento"] = (
    X["congelado_general"].shift(1, fill_value=False) & ~X["congelado_general"]
)  # marca donde los sensores se descongelan

train, test = train_test_split(X, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)
train = train.sort_values(COL_TIEMPO).set_index(COL_TIEMPO)
n_test, rango_test = len(test), (test[COL_TIEMPO].min(), test[COL_TIEMPO].max())
del test  # a partir de aquí el test no se vuelve a usar

print(f"train: {len(train):,} filas · test (apartado): {n_test:,} filas")
print(f"train cubre {train.index.min()} → {train.index.max()}")

## 3. Validación de esquema (train)

Se compara contra el contrato esperado: columnas, tipos, timestamps parseables(formato correcto) y únicos, valores numéricos no negativos, y acumulados que no decrecen.

In [ ]:
def chequeo(nombre: str, ok: bool, detalle: str) -> dict[str, str]:
    """Arma una fila de la tabla de validación de esquema."""
    return {"chequeo": nombre, "estado": "PASA" if ok else "FALLA", "detalle": detalle}


esperadas, presentes = set(ESQUEMA), set(raw.columns)
checks = [
    chequeo(
        "Columnas esperadas presentes",
        esperadas <= presentes,
        f"faltan: {sorted(esperadas - presentes) or '—'}",
    ),
    chequeo(
        "Sin columnas inesperadas",
        presentes <= esperadas,
        f"sobran: {sorted(presentes - esperadas) or '—'}",
    ),
    chequeo(
        "t_stamp formato correcto",
        not train[f"cast_falla__{COL_TIEMPO}"].any(),
        f"{int(train[f'cast_falla__{COL_TIEMPO}'].sum())} fallos",
    ),
    chequeo(
        "t_stamp sin nulos", train.index.notna().all(), f"{int(train.index.isna().sum())} nulos"
    ),
    chequeo(
        "t_stamp único",
        not train.index.duplicated().any(),
        f"{int(train.index.duplicated().sum())} duplicados",
    ),
]
for m in MEDIDORES:
    fallos = int(train[f"cast_falla__{m}"].sum())
    negativos = int((train[m] < 0).sum())
    decrece = int((train[f"delta__{m}"] < 0).sum())
    checks += [
        chequeo(f"{m}: numérico", fallos == 0, f"{fallos} celdas no convertibles"),
        chequeo(f"{m}: lectura ≥ 0", negativos == 0, f"{negativos} negativos"),
        chequeo(f"{m}: acumulado no decreciente", decrece == 0, f"{decrece} filas con delta < 0"),
    ]
esquema = pd.DataFrame(checks)
display(
    esquema.style.map(
        lambda v: "color:#b91c1c;font-weight:bold" if v == "FALLA" else "color:#15803d",
        subset=["estado"],
    )
)
print(f"Resultado: {(esquema.estado == 'PASA').sum()} de {len(esquema)} chequeos pasan")

Todos los medidores fallaron la prueba de "acumulado no decreciente".

Esto significa que, aunque las lecturas son positivas, en ciertos momentos el medidor marca un número menor que en la lectura anterior (lo que hace que el delta sea negativo).

¿Por qué suele pasar esto?

- Reinicio del medidor (Rollover): El contador llegó a su límite máximo y volvió a empezar desde cero.

- Mantenimiento: Cambiaron el sensor físico por uno nuevo o lo resetearon tras una calibración.

- Picos de error (Outliers): Un ruido en el SCADA mandó un número bajo y luego volvió a la normalidad.

## 4. Consistencia de tipos y conversiones necesarias

In [ ]:
tipos = pd.DataFrame(
    {
        "dtype_en_archivo": dtypes_crudos,
        "dtype_esperado": pd.Series(ESQUEMA),
        "dtype_tras_cast": train.reset_index()[list(ESQUEMA)].dtypes.astype(str),
    }
)


def _cumple(dtype_txt: str, esperado: str) -> bool:
    """Indica si el dtype leído coincide con el tipo lógico esperado."""
    if esperado == "datetime":
        return dtype_txt.startswith("datetime64")
    return dtype_txt == esperado


tipos["requiere_cast"] = [
    not _cumple(a, e)
    for a, e in zip(tipos["dtype_en_archivo"], tipos["dtype_esperado"], strict=True)
]
display(tipos)

# ¿Los valores vienen de un float32? (granularidad en saltos de 2^k por la precisión)
prec = []
for m in MEDIDORES:
    v = train[m].dropna()
    v = v[v > 0]
    es_f32 = np.isclose(v.astype(np.float32).astype(np.float64), v, rtol=0, atol=1e-6).mean()
    entero = np.isclose(v % 1, 0).mean()
    paso = np.spacing(np.float32(v.max())) if len(v) else np.nan
    prec.append(
        {
            "medidor": m,
            "% representable exacto en float32": es_f32 * 100,
            "% valores enteros": entero * 100,
            "resolución float32 en el máximo": float(paso),
            "máximo": v.max(),
        }
    )
display(pd.DataFrame(prec).set_index("medidor"))

Todas las columnas dicen False en requiere_cast (no se tuvo que forzar transformaciones para arreglar el tipo de dato.). Esto significa que el archivo Excel viene bien estructurado. Las fechas son reconocidas como fechas (datetime64) y los consumos como números reales (float64).

## **Lectura:**
- Si el archivo trae algún tipo distinto al esperado, la tabla lo marca en `requiere_cast`. El pipeline de modelado debe convertir explícitamente `t_stamp → datetime64` y medidores `→ float64` con `errors="coerce"`, y tratar los fallos como faltantes.


La columna `resolución float32 en el máximo` Es el 'salto mínimo' que da el contador cuando los números son muy grandes; si la resolución dice '512', significa que el medidor ya no puede contar de 1 en 1, sino que salta de golpe de 512 en 512, perdiéndose todos los consumos más pequeños.

## ***¿Por qué hay consumos negativos en los datos?***

- Límite de memoria: El SCADA guarda los datos en float32.

- Pérdida de precisión: Al acumular números tan grandes, este formato ya no puede registrar cambios pequeños y empieza a hacer redondeos bruscos.

- Falsos negativos: Las pequeñas caídas en el consumo no son reales. El medidor no retrocede físicamente; estas anomalías son simples errores de cálculo causados por el problema matemático de redondeo del sistema

- Presencia de valores interpolados: Se encontró una pequeña proporción (0.5% - 1.5%) de valores con decimales en medidores que normalmente reportan números enteros. Esto sugiere que el historiador del SCADA realizó interpolaciones para rellenar huecos en la transmisión de datos; por tanto, esas lecturas específicas son reconstrucciones del sistema, no mediciones reales del equipo.

- Las columnas de "acumulado no decreciente" fallan en el esquema por los reinicios, los negativos de cuantización y la última fila en cero.

In [ ]:
# ¿Cuáles son los falsos negativos más comunes?
print("Top de valores negativos (saltos) más repetidos por medidor:")
for m in MEDIDORES:
    # Filtramos solo las filas donde el delta es menor a 0
    deltas_neg = train.loc[train[f"delta__{m}"] < 0, f"delta__{m}"]

    if not deltas_neg.empty:
        print(f"\n▶ {m}:")
        # Mostramos los 3 valores negativos que más se repiten
        display(deltas_neg.value_counts().head(3).to_frame(name="Cantidad de veces"))

print("-" * 50)

# Lupa
medidor_ejemplo = "FILTRO 2"
columnas_ver = [f"prev__{medidor_ejemplo}", medidor_ejemplo, f"delta__{medidor_ejemplo}"]

# Filtramos las filas donde ese medidor dio negativo
ejemplos = train[train[f"delta__{medidor_ejemplo}"] < 0][columnas_ver]

print(f"\n🔍 Ejemplo visual de cómo ocurre el salto en '{medidor_ejemplo}':")
display(ejemplos.head())


## 5. Valores faltantes: conteos y patrones

En un contador acumulado, el faltante **no** se ve como `NaN`. Se definen estas categorías (**solo en train**):

| Categoría | Regla |
|---|---|
| `nan_archivo` | celda vacía o no convertible |
| `no_integrado` | lectura en 0 antes de la primera lectura válida del medidor (medidor aún no conectado al SCADA) |
| `sin_lectura` | lectura en 0 (centinela) o > 100 × mediana del medidor (fuera de escala) |
| `previa_invalida` | la lectura anterior es 0 / NaN / fuera de escala → el delta no es interpretable |
| `congelado` | todos los medidores de proceso sin cambio a la vez (SCADA sin actualizar) |
| `recuperacion` | primer delta después de un congelamiento (acumula lo no registrado) |
| `reinicio` | delta negativo mayor que la tolerancia |
| `ruido_neg` | delta negativo dentro de la tolerancia (cuantización float32) |
| `salto` | delta > 5 × p99.5 del medidor (pico alto NO después de congelamiento) |
| `ok` | consumo válido |

In [ ]:
L = train[MEDIDORES]  # lecturas actuales
P = train[[f"prev__{m}" for m in MEDIDORES]].set_axis(MEDIDORES, axis=1)  # lecturas previas
D = train[[f"delta__{m}" for m in MEDIDORES]].set_axis(MEDIDORES, axis=1)  # deltas
CF = train[[f"cast_falla__{m}" for m in MEDIDORES]].set_axis(MEDIDORES, axis=1)  # fallos de formato

# --- parámetros aprendidos SOLO de train ---
mediana_lectura = L.where(L > 0).median()  # calcula el valor acumulado "normal" de cada medidor
validas = (L > 0) & (P > 0) & L.le(100 * mediana_lectura) & P.le(100 * mediana_lectura)
mediana_delta_pos = D.where(validas & (D > 0)).median()
TOL_NEG = 0.05 * mediana_delta_pos  # si la caida es menor a el 5%
# de la mediana de los deltas positivos, se considera ruido y no reinicio
UMBRAL_SALTO = 5 * D.where(validas & (D >= 0)).quantile(0.995)  # consumo máximo aceptable
primera_valida = L.where(L > 0).apply(
    lambda s: s.first_valid_index()
)  # fecha en el que despertó cada medidor
PARAMS = pd.DataFrame(
    {
        "mediana_lectura": mediana_lectura,
        "mediana_delta_pos": mediana_delta_pos,
        "tolerancia_neg": TOL_NEG,
        "umbral_salto": UMBRAL_SALTO,
        "primera_lectura_valida": primera_valida,
    }
)
display(PARAMS)

fuera = L.gt(100 * mediana_lectura)
prev_inv = P.isna() | P.eq(0) | P.gt(100 * mediana_lectura)
cong = pd.DataFrame(dict.fromkeys(MEDIDORES, train["congelado_general"]))
tras = pd.DataFrame(dict.fromkeys(MEDIDORES, train["tras_congelamiento"]))
antes_integ = pd.DataFrame(
    {m: train.index < primera_valida[m] for m in MEDIDORES}, index=train.index
)

flag = pd.DataFrame("ok", index=train.index, columns=MEDIDORES)
reglas = [  # se aplican en orden inverso de prioridad: la última que coincide gana
    ("salto", D.gt(UMBRAL_SALTO)),
    ("ruido_neg", D.lt(0) & D.ge(-TOL_NEG)),
    ("reinicio", D.lt(-TOL_NEG)),
    ("recuperacion", tras & D.gt(UMBRAL_SALTO)),
    ("congelado", cong),
    ("previa_invalida", prev_inv),
    ("sin_lectura", L.eq(0) | fuera),
    ("no_integrado", L.eq(0) & antes_integ),
    ("nan_archivo", L.isna() | CF),
]
for nombre, mascara in reglas:
    flag = flag.mask(mascara, nombre)

ORDEN_FLAGS = [
    "ok",
    "ruido_neg",
    "congelado",
    "recuperacion",
    "reinicio",
    "salto",
    "previa_invalida",
    "sin_lectura",
    "no_integrado",
    "nan_archivo",
]
conteo = flag.apply(pd.Series.value_counts).reindex(ORDEN_FLAGS).fillna(0).astype(int)
pct = conteo / len(flag) * 100
display(conteo)
display(pct.round(2).rename(index=lambda s: f"% {s}"))

In [ ]:
largo = flag.melt(var_name="medidor", value_name="categoria")
fancy_bars(
    largo[largo.categoria != "ok"],
    "categoria",
    labels="count",
    orient="h",
    figsize=(9, 5),
    title="Celdas no válidas por categoría (train)",
)
plt.show()

### 5.1 Patrones: ¿cuándo faltan los datos?

Un mapa por mes muestra si el faltante es estructural (por ejemplo, medidores no integrados), si llega en ráfagas (congelamientos) o si está disperso.

In [ ]:
invalido = ~flag.isin(["ok"])
mapa = invalido.groupby(invalido.index.to_period("M")).mean() * 100
fig, ax = plt.subplots(figsize=(16, 4.8))
sns.heatmap(mapa.T, cmap="Reds", vmin=0, vmax=100, ax=ax, cbar_kws={"label": "% filas no válidas"})
ax.set_title(
    "% de filas no válidas por medidor y mes (train)", loc="left", fontweight="bold", color=INK
)
ax.set_xticklabels([str(p) for p in mapa.index], rotation=90, fontsize=8)
plt.show()

co = invalido.astype(int).corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(co, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title(
    "Co-ocurrencia de faltantes (corr. de indicadores)", loc="left", fontweight="bold", color=INK
)
plt.show()

- EMCALI y EDIF ADMON fallan al mismo tiempo, mientras que las otras máquinas del proceso (filtros, hornos, refinos) fallan juntas.

- El mapa de calor nos muestra una franja roja para EMCALI y EDIF ADMON desde el inicio de los datos hasta aproximadamente febrero-marzo del 2026, lo que nos puede mostrar que los medidores no estaban dañados, sino que fueron instalados mucho después que el resto

- También podemos ver que al momento de fallar, no se dañan de forma individual, lo podemos ver con la franja vertical naranja que atraviesa todos los medidores de proceso, esto nos muestra una desconexión general del SCADA o un mantenimiento en donde los sensores dejaron de reportar a la vez

In [ ]:
# Rachas de congelamiento general en train (filas train consecutivas en el tiempo real)
c = train["congelado_general"]
contiguo = train["paso_h"].round(2).eq(1)  # la fila anterior real también está en train
bloque = (~(c & c.shift(fill_value=False) & contiguo)).cumsum()
rachas = (
    train[c]
    .groupby(bloque[c])
    .apply(
        lambda g: pd.Series(
            {
                "inicio": g.index.min(),
                "fin": g.index.max(),
                "horas_aprox": (g.index.max() - g.index.min()).total_seconds() / 3600 + 1,
            }
        )
    )
)
rachas = rachas.sort_values("horas_aprox", ascending=False).reset_index(drop=True)
display(rachas.head(8))
print("horas_aprox = cuanto duró el 'apagón")

# chi-cuadrado del indicador de faltante de proceso contra mes, día y hora
falta_proc = invalido[PROCESO].any(axis=1)
filas = []
for nombre, cat in {
    "mes": train.index.to_period("M").astype(str),
    "dia_semana": train.index.dayofweek,
    "hora": train.index.hour,
}.items():
    tabla = pd.crosstab(cat, falta_proc)
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    v = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))
    filas.append(
        {
            "faltante vs": nombre,
            "chi2": chi2,
            "gl": gl,
            "p_valor": p,
            "V de Cramér": v,
            "¿caidas aleatorias?": "NO" if p < ALPHA else "SI",
        }
    )
display(pd.DataFrame(filas))

**Lectura final:**

El análisis de datos faltantes revela tres comportamientos distintos.

- EMCALI y EDIF ADMON faltan de forma estructural antes de mar-2026 (`no_integrado`). No hay nada que imputar, la variable simplemente no existía.

- Las pérdidas de información en la planta ocurren en apagones simultáneos para todas las máquinas, confirmando que es una falla de conexión del SCADA y no sensores dañados. Estos apagones no son aleatorios: estadísticamente se concentran en ciertos meses del año, aunque no siguen un patrón de días u horas. 

- Los pequeños consumos negativos (ruido_neg) ocurren de forma aislada en cada máquina debido a los problemas de redondeo matemático del sistema, por lo que no representan una pérdida real de información, sino un simple ruido a filtrar.



## 6. Integridad temporal y frecuencia (train)

`paso_h` se calculó antes de dividir los datos, mantiene el tiempo real entre mediciones físicas, evitando errores aunque la fila anterior haya quedado en el set de test.

In [ ]:
# se realiza un chequeo de "salud" sobre la linea del tiempo
# para asegurar una estructura temporal solida
paso = train["paso_h"]
print("revisión de integridad temporal del dataset:")
integridad = pd.Series(
    {
        "inicio": train.index.min(),
        "fin": train.index.max(),
        "orden cronológico (tras ordenar)": train.index.is_monotonic_increasing,
        "timestamps duplicados": int(train.index.duplicated().sum()),
        "paso mediano (h)": round(paso.median(), 4),
        "% filas con paso ~ 1 h": round(paso.round(2).eq(1).mean() * 100, 2),
        "filas con paso < 0,95 h": int((paso < PASO_MIN_H).sum()),
        "filas con paso > 1,5 h (huecos)": int((paso > PASO_MAX_H).sum()),
        "zona horaria": "naive (sin tz) — asumir America/Bogota, sin horario de verano",
    }
)
display(integridad.to_frame("valor"))

print("Minuto de la lectura (deriva del reloj del SCADA):")
display(train.index.minute.value_counts().head(6).to_frame("filas").T)

cobertura = train.groupby(train.index.to_period("M")).size().to_frame("filas_train")
horas_mes = cobertura.index.map(lambda p: p.days_in_month * 24)
cobertura["% de horas del mes en train"] = cobertura["filas_train"] / horas_mes * 100
display(cobertura.describe().T)

**Lectura final:**

La serie abarca desde octubre del 2023 hasta octubre del 2026, estando ordenada y sin fechas duplicadas. Tamboén podemos observar que el 99.98% de las filas tienen 1 hora de diferencia entre sí, sin huecos mayores a 1.5h. Los minutos de la lectura muestran que la mayor parte de las filas están en los minutos 30-31, que el SCADA está programado para hacer un "barrido" de datos de datos de forma automática a la mitad de cada hora. Se pueden ver unos pequeños desfases hacia 15, 01 y 46, debido a estos pequeños desplazamientos en los minutos, el pipeline deberá alinear los datos a una grilla horaria exacta (floor('h') o resample) antes de unirlos con otras fuentes. Finalmente, la media de cobertura mensual ronda el ~80%, lo cual tiene sentido con el split realizado de train-test; esto asegura que cada mes conserve suficientes datos y la estacionalidad quede bien representada para el modelo.


## 7. Estacionalidad (train)

Se busca descubrir los ritmos y patrones del consumo de la planta, para revisar en qué momentos se consume más energía y cuando se tienen picos.

Para ello, se usa el consumo horario **válido** (`ok`) del total de los medidores de proceso (`TOTAL_PROCESO`, disponible en todo el periodo). El turno se asigna según el punto medio del intervalo.



In [ ]:
cons_ok = D.where(flag.eq("ok"))
ts = pd.DataFrame(index=train.index)
ts["TOTAL_PROCESO"] = cons_ok[PROCESO].sum(axis=1, min_count=len(PROCESO))
t_mid = train.index.to_series() - pd.to_timedelta(train["paso_h"].fillna(1) / 2, unit="h")
ts["hora"] = t_mid.dt.hour
ts["turno"] = pd.Categorical(
    np.select([ts.hora.ge(22) | ts.hora.lt(6), ts.hora.lt(14)], ["T1", "T2"], "T3"),
    ["T1", "T2", "T3"],
)
ts["dia_semana"] = pd.Categorical(t_mid.dt.dayofweek.map(dict(enumerate(DIAS))), DIAS)
ts["mes"] = t_mid.dt.month
ts["anio_mes"] = t_mid.dt.to_period("M")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (col, titulo) in zip(
    axes,
    [
        ("hora", "por hora del día"),
        ("dia_semana", "por día de la semana"),
        ("mes", "por mes del año"),
    ],
    strict=True,
):
    g = ts.groupby(col, observed=True)["TOTAL_PROCESO"]
    ax.set_ylabel("Consumo medio por hora (kWh)", fontsize=9)
    m, s = g.mean(), g.sem() * 1.96
    ax.errorbar(
        range(len(m)),
        m.values,
        yerr=s.values,
        marker="o",
        ms=4,
        color="#5BA7DE",
        ecolor="#94a3b8",
        capsize=2,
    )
    ax.set_xticks(range(len(m)))
    ax.set_xticklabels(m.index.astype(str), fontsize=8)
    ax.set_title(
        f"TOTAL_PROCESO medio {titulo}", loc="left", fontweight="bold", color=INK, fontsize=11
    )
    ax.grid(axis="y", color=GRID, ls="--")
    sns.despine(ax=ax, left=True)
for h in (6, 14, 22):
    axes[0].axvline(h, color="#E28585", ls=":", lw=1)
plt.tight_layout()
plt.show()

**lectura por hora:**

- Podemos obseervar que el turno de noche es el que tiene menor consumo de energía, rondando entre los 380.000-400.000 kWh, mientras que el que más tiene s el de la mañana
- Hay un pico a las 6a.m posinlemente de arranque, manteniendo el consumo muy alto toda la mañana (casi siempre por encima de 420.000kWh)
- Hay una caida pronunciada a la 1p.m (horario que normalmente es una hora de almuerzo), en donde varias máquinas bajan su ritmo o pausan su funcionamiento
- Hay caidas de consumo antes de cada final de jornada.

**lectura por día**
- El domingo es el día que menor consumo operativo tiene, cayendo a los 390.000kWh
- El lunes inicia con un consumo moderado y a medida que avanza la semana, la planta alcanza su máxima caapcidad de producción, más específicamente entre los días miércoles y jueves, superando los 415.000 kWh

**lectura por mes**
- Noviembre es el mes con mayor exigencia para la planta, con el pico más alto casi llegando a 460.000kWh
- Diciembre es el mes con el punto más bajo de todo el año cerca de los 320.000kWh, lo cuál tiene sentido por vacaciones y festividades de fin de año, o, por posibles mantenimientos.
- También se ve una caida en el consumo en los meses de marzo y abril, lo que también suele coincidir con meses con varios festivos (ej:semana santa)

In [ ]:
# Autocorrelación sobre la grilla horaria (las horas de test y no válidas quedan como NaN)
grilla = ts["TOTAL_PROCESO"].copy()
grilla.index = grilla.index.floor(
    "h"
)  # deja las horas en formato datetime, sin minutos ni segundos
grilla = grilla[~grilla.index.duplicated()].asfreq(
    "h"
)  # se rellena con NaN las horas que no están en train (test o huecos)
r = acf(
    grilla, nlags=24 * 8, missing="conservative"
)  # correlación de la serue con respecto a sí misma, hasta 8 días (24 h * 8)
fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(range(len(r)), r, color="#5BA7DE", width=0.8)
for k in (24, 48, 72, 96, 120, 144, 168):
    ax.axvline(k, color="#E28585", ls=":", lw=1)
ax.set_title(
    "ACF del consumo horario (líneas = múltiplos de 24 h; 168 h = 1 semana)",
    loc="left",
    fontweight="bold",
    color=INK,
)
ax.set_xlabel("rezago (h)")
sns.despine(left=True)
plt.show()
display(pd.Series({f"lag {k} h": r[k] for k in (1, 8, 12, 24, 168)}).to_frame("ACF").T)

In [ ]:
pruebas = hub_pruebas_num_cat(ts, "TOTAL_PROCESO", ["turno", "dia_semana", "mes"])


def eps2(cat: str) -> float:
    """Tamaño del efecto ε² de Kruskal-Wallis del consumo frente a `cat`."""
    sub = ts[["TOTAL_PROCESO", cat]].dropna()
    h = stats.kruskal(
        *[g.values for _, g in sub.groupby(cat, observed=True)["TOTAL_PROCESO"]]
    ).statistic
    return h / (len(sub) - 1)


pruebas["epsilon2"] = pruebas["var_categorica"].map(eps2)
display(pruebas[["var_categorica", "n_grupos", "prueba", "p_valor", "epsilon2", "decision"]])
display(
    ts.groupby("turno", observed=True)["TOTAL_PROCESO"]
    .agg(["count", "mean", "median"])
    .assign(horario=lambda d: d.index.map(TURNOS))
)

**Lectura final:**
- **Ciclo diario por turnos:** el consumo sube en T2 (07–12 h) y cae justo antes de cada relevo (05, 13 y 21 h).

- **Ciclo semanal:** domingo y lunes son los días más bajos.

- **Estacionalidad anual o de régimen:** marzo, abril y diciembre son claramente más bajos (paradas y campañas del horno).

- **ACF:** el consumo no da saltos bruscos: lo que la planta gasta en una hora se parece mucho a lo que gastó la hora anterior (lag 1h~ 0,72). Además se ven repuntes moderados cada 24 h y cada 168 h (una semana), que reflejan la rutina diaria y semanal de la planta. Como el nivel de consumo sube y baja con las campañas, la serie no es estacionaria. Para el modelo conviene usar información del pasado reciente ("ayer a esta hora", "hace una semana") junto con variables de calendario (turno, día de la semana y mes).

- **Mes:** Es el factor más importante; Al medir matemáticamente qué condiciones alteran más el ritmo de producción, el mes del año resultó ser el más decisivo, sin embargo sigue siendo un efecto bastante débil.



- **Tamaño del efecto (ε²):** de las tres variables de calendario, el mes es la que más diferencia el consumo. Según las convenciones habituales (0,01 pequeño, 0,06 moderado, 0,14 grande), el efecto del mes es moderado; sin embargo, explica solo ~8 % de la variabilidad del consumo, de modo que la mayor parte depende de factores operativos (campañas del horno, paradas) más que del calendario. Las pruebas son significativas, pero con tantos datos eso ocurre casi siempre; por eso se compara con el tamaño del efecto.

## 8. Distribuciones y outliers (train, solo consumos `ok`)

In [ ]:
desc = cons_ok.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["asimetria"] = cons_ok.skew()
desc["curtosis"] = cons_ok.kurt()
desc["% en cero"] = cons_ok.eq(0).sum() / cons_ok.notna().sum() * 100
display(desc)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, m in zip(axes.flat, MEDIDORES, strict=True):
    fancy_hist(cons_ok, m, ax=ax, title=m)
    ax.title.set_fontsize(12)
plt.suptitle(
    "Consumo horario válido por medidor (train)",
    x=0.01,
    ha="left",
    fontsize=16,
    fontweight="bold",
    color=INK,
)
plt.tight_layout()
plt.show()

In [ ]:
# Outliers: regla de Tukey (1,5·IQR) y z robusto (MAD > 3,5)
out = []
for m in MEDIDORES:
    s = cons_ok[m].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    mad = stats.median_abs_deviation(s, scale="normal")
    z = (s - s.median()) / mad if mad > 0 else pd.Series(0, index=s.index)
    out.append(
        {
            "medidor": m,
            "n": len(s),
            "% Tukey bajo": (s < q1 - 1.5 * iqr).mean() * 100,
            "% Tukey alto": (s > q3 + 1.5 * iqr).mean() * 100,
            "% |z_MAD| > 3,5": (z.abs() > Z_MAD_UMBRAL).mean() * 100,
            "máx / p99": s.max() / s.quantile(0.99),
        }
    )
outliers = pd.DataFrame(out).set_index("medidor")
display(outliers)

fig, ax = plt.subplots(figsize=(14, 4.5))
norm = cons_ok[PROCESO] / cons_ok[PROCESO].median()
sns.boxplot(data=norm, ax=ax, color="#5BA7DE", fliersize=1.5, linewidth=1)
ax.set_title(
    "Consumo horario / mediana de cada medidor (train)", loc="left", fontweight="bold", color=INK
)
ax.set_yscale("symlog")
sns.despine(left=True)
plt.show()

In [ ]:
rep = normality_report(ts, "TOTAL_PROCESO")

**Lectura:**


- Las distribuciones son multimodales (operación / parada) y con asimetría negativa en el total:

    * HORNO: Su percentil 25% es de apenas 7,680, pero su mediana salta a 101,120. Esta enorme diferencia comprueba que la máquina pasa al menos una cuarta parte del tiempo apagada o al mínimo, y luego opera a niveles altísimos.

    * TRITURADO: Su desviación estándar supera a su propio promedio y tiene el boxplot más ancho de todos. Su histograma muestra varias crestas, lo que refleja los constantes cambios de intensidad a lo largo de los distintos turnos.

    * Filtros: Representan la carga base de la planta, manteniendo una distribución estrecha y un consumo constante.

- Los "outliers bajos" son mayormente paradas reales y no errores, y los altos extremos son pocos una vez se excluyen saltos y recuperaciones. No conviene recortarlos ya que son la realidad física de la planta. En el modelado, mejor representarlos con un indicador de estado (encendido/apagado).

- El total no es normal, así que conviene usar estadísticos robustos y pruebas no paramétricas.


### 8.1 Relaciones entre medidores (bivariado, train)

Se compara el consumo horario medio por día de cada medidor (solo horas `ok` de train). Se usa la media y no la suma porque, con el split aleatorio, cada día tiene ~80 % de sus horas en train. Spearman mide relación monótona y Pearson relación lineal; si difieren mucho, la relación no es lineal o la dominan los regímenes de encendido/apagado.

In [ ]:
dia_ok = (
    train.index.to_series()
    .sub(pd.to_timedelta(train["paso_h"].fillna(1) / 2, unit="h"))
    .dt.normalize()
)
g = cons_ok[PROCESO].groupby(dia_ok)
diario = g.mean().where(
    g.count() >= MIN_HORAS_VALIDAS_DIA
)  # mínimo 12 horas válidas por día y medidor
n_completos = diario.dropna().shape[0]
print(f"Días con datos: {len(diario):,} · días completos (7 medidores): {n_completos:,}")

corr_s = diario.corr(method="spearman")
corr_p = diario.corr(method="pearson")
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
mask = np.triu(np.ones_like(corr_s, dtype=bool), 1)
cmap = sns.diverging_palette(15, 210, s=75, l=60, as_cmap=True)
for ax, c, nombre in zip(axes, [corr_s, corr_p], ["Spearman", "Pearson"], strict=True):
    sns.heatmap(
        c,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        vmin=-1,
        vmax=1,
        center=0,
        linewidths=1,
        ax=ax,
        cbar=False,
    )
    ax.set_title(
        f"Correlación de {nombre} entre medidores (consumo medio diario, train)",
        loc="left",
        fontweight="bold",
        color=INK,
        fontsize=11,
    )
plt.tight_layout()
plt.show()

In [ ]:
pares = (
    corr_s.where(np.triu(np.ones_like(corr_s, dtype=bool), 1)).stack().rename("spearman").to_frame()
)
pares["pearson"] = corr_p.stack().reindex(pares.index)
pares["dif_s_p"] = pares["spearman"] - pares["pearson"]
pares = pares.reindex(pares["spearman"].abs().sort_values(ascending=False).index)
pares.index.names = ["medidor_a", "medidor_b"]
display(pares.head(10))

top = pares.index[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (a, b) in zip(axes, top, strict=True):
    ax.scatter(diario[a], diario[b], s=6, alpha=0.35, color="#5BA7DE")
    ax.set_xlabel(a)
    ax.set_ylabel(b)
    ax.set_title(
        f"rho = {pares.loc[(a, b), 'spearman']:.2f}", loc="left", fontweight="bold", color=INK
    )
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# TOTAL_PROCESO se usa como "target" solo para excluirlo del cálculo.
# Sin cache_path → no escribe archivos.
diario_vif = diario.dropna().assign(TOTAL_PROCESO=lambda d: d.sum(axis=1))
vif = vif_table(diario_vif, target_variable="TOTAL_PROCESO", feature_variables=PROCESO)
display(vif)

**Lectura final:**

- Las correlaciones entre medidores son moderadas (la mayor es REFINO 1 con REFINO 2, con ρ ~ 0.55). Esto indica que cada área de la planta tiene su propia dinámica operativa.

- En FILTRO 2 – HORNO Pearson es alto (~ 0.76) pero Spearman cae (~ 0.47). La relación lineal es engañosa y es generada por los estados de encendido/apagado (cuando el horno se apaga, el filtro baja con él), y no porque mantengan una proporción constante mientras operan.

- REFINO 2 Tiene un coeficiente de Spearman mucho mayor que el de Pearson con REFINO 1 y FILTRO 1. Esto significa que acompaña la operación general de la planta, pero presenta picos de consumo que rompen la linealidad.

- El Factor de Inflación de Varianza (VIF) es menor a 3 en todos los medidores. No hay información redundante grave, por lo que ninguna variable sobra para el modelado.

- TOTAL_PROCESO es la suma exacta de los 7 medidores. un modelo no debe incluir el total junto con todos sus componentes por la colinealidad perfecta.

## 9. Salud general de los datos (train)

In [ ]:
salud = pd.DataFrame(
    {
        "% válido (ok)": (flag == "ok").mean() * 100,
        "% faltante estructural": flag.isin(["no_integrado"]).mean() * 100,
        "% faltante operativo": flag.isin(
            [
                "nan_archivo",
                "sin_lectura",
                "previa_invalida",
                "congelado",
                "recuperacion",
                "reinicio",
                "salto",
            ]
        ).mean()
        * 100,
        "% ruido de cuantización": (flag == "ruido_neg").mean() * 100,
        "% outliers Tukey (sobre ok)": outliers["% Tukey bajo"] + outliers["% Tukey alto"],
    }
)
salud["estado"] = np.select(
    [
        salud["% faltante estructural"] > PCT_ESTRUCTURAL,
        salud["% faltante operativo"] > PCT_REVISAR,
        salud["% faltante operativo"] > 1,
    ],
    ["usable solo desde su integración", "revisar", "aceptable"],
    "bueno",
)
display(salud)
print(
    f"Chequeos de esquema que fallan: {(esquema.estado == 'FALLA').sum()} → "
    + ", ".join(esquema.loc[esquema.estado == "FALLA", "chequeo"])
)

## 10. Estrategia de manejo de faltantes (documentada, **no aplicada**)

| Problema en los datos | Causa probable | Solución principal (Tratamiento) | Plan B (Alternativas del modelo) |
| :--- | :--- | :--- | :--- |
| **Texto ilegible en Excel** (`nan_archivo`) | Error al exportar el archivo original desde la fuente. | Forzar la conversión del error a un dato vacío (NaN) y tratarlo como una lectura faltante. | N/A |
| **Medidores desconectados al inicio** (`no_integrado`) | Los medidores de EMCALI y Administración no existían o no estaban conectados antes de marzo 2026. | No inventar ni rellenar datos. Recortar el periodo del modelo a las fechas donde sí existen, o modelar esas áreas por separado. | Si se requiere analizar toda la historia desde el inicio, usar solo los medidores de las máquinas principales (proceso). |
| **Lectura faltante** (`sin_lectura` / `previa_invalida`) | Falla momentánea de comunicación con el medidor. | Dejar el consumo vacío (NaN) y crear una columna indicadora de "dato faltante". | Si el hueco es muy corto (máximo 2 horas), interpolar linealmente el acumulado (no el consumo) y recalcular la diferencia. |
| **SCADA congelado y salto de recuperación** (`congelado` + `recuperacion`) | El SCADA dejó de actualizar todos los medidores a la vez (MAR respecto al periodo) y luego registró de golpe el consumo acumulado. | Dejar vacías las horas congeladas y crear un indicador. Para totales, repartir el salto equitativamente entre las horas congeladas (así se conserva la energía). | Para el modelo: excluir ese bloque o rellenarlo con el perfil típico hora × día de la semana aprendido en train, escalado para que sume el total recuperado. |
| **Contador reiniciado** (`reinicio`) | Cambio del medidor o reinicio del contador. | Dejar vacía esa lectura (NaN). Para mantener continuo el acumulado, sumarle el último valor registrado antes del reinicio. | N/A |
| **Consumo negativo pequeño** (`ruido_neg`) | Límite de precisión del formato float32 con números muy grandes (cuantización). | Llevar el valor a 0, porque no es posible consumir energía negativa. | N/A |
| **Picos anómalos** (`salto`) | Lectura errónea del sistema (valor fuera de rango). | Dejar el valor vacío (NaN) y crear un indicador de salto. No recortarlo ni suavizarlo. | Si se puede, validar si ocurrió un evento extraordinario. |

- Al aplicar matemáticas para rellenar huecos (como interpolaciones), está estrictamente prohibido usar información que ocurra después de la fecha de corte. El modelo no puede usar datos del conjunto de prueba para "arreglar" el conjunto de entrenamiento.

- Las columnas que avisan que hubo un error o un dato vacío no se eliminan. Se conservan como variables oficiales, ya que enseñan al modelo a reaccionar ante los apagones o anomalías operativas de la planta.

Boceto de interfaz:
```python
class LimpiezaContadores(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):      # aprende medianas, tolerancias y umbrales SOLO con train
        ...
    def transform(self, X):        # deltas, categorías, indicadores; sin reajustar
        ...
```

## 11. Hallazgos principales (train: 21.044 filas, 80 %)

1. **Estructura sana:** la base tiene las columnas esperadas, con los tipos correctos y sin registros duplicados. Los únicos fallos del esquema vienen de que el acumulado a veces baja por reinicios del contador, pequeños negativos por precisión numérica y una última fila en cero.
2. **Precisión numérica:** el SCADA guarda las lecturas en `float32`, que con números tan grandes pierde precisión. El pipeline debe convertir a `float64` y tolerar esos pequeños desajustes para no confundirlos con errores reales.
3. **Por qué faltan datos:**
   * **Medidores nuevos:** a EMCALI y EDIF ADMON les falta ~82 % de la historia porque no estaban conectados al SCADA antes de marzo de 2026 (faltante estructural).
   * **Congelamientos del SCADA:** los medidores de proceso perdieron ~2,6 % de sus datos por periodos en que el SCADA dejó de actualizar todos los medidores a la vez (el más largo, entre julio y agosto de 2025). El faltante depende del periodo y no de la hora (MAR).
4. **Desfase del reloj:** hay una lectura por hora, pero el minuto de registro cambia (de :31 a :30, y en las últimas filas :15, :01 y :46). Hay que alinear todos los registros a una grilla horaria.
5. **Patrones de consumo:** el consumo baja los domingos y lunes, en marzo, abril y diciembre, y justo antes de cada cambio de turno (05, 13 y 21 h).
6. **Los extremos no son errores:** las caídas a valores cercanos a cero, sobre todo en los filtros, son paradas reales y no deben borrarse. Solo los saltos extremos se marcan como `salto` o `recuperacion`.
7. **Ningún medidor sobra:** todos tienen VIF < 3, así que no hay redundancia. La relación entre medidores depende más de si se encienden y apagan juntos que de una proporción fija.